# 门店销量预测与异常监控报告**项目目标**：完成复杂的销量预测任务，进行异常检测，整合所有分析结果，撰写最终结项报告。| 模块 | 内容 ||------|------|| 数据探索 | 加载 `data3.csv`，理解门店、价格、销量结构 || 影响因素分析 | 节假日效应、季度季节性对销量的影响 || 异常检测 | 销量与单价的离群值识别与监控 || 销量预测 | 多模型对比，跨门店横向比较 || 结项报告 | 综合结论与业务建议 |

In [ ]:
# -*- coding: utf-8 -*-
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, IsolationForest
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'PingFang SC', 'Heiti TC']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

DATA_PATH = 'data3.csv'
REPORT_DATE = datetime.now().strftime('%Y-%m-%d')
print('环境就绪，报告日期:', REPORT_DATE)


---## 一、数据加载与预处理

In [ ]:
df_raw = pd.read_csv(DATA_PATH, encoding='gbk')
df = df_raw.copy()
df.columns = ['商品ID', '类别ID', '门店编号', '单价', '销量', '成交时间', '订单ID']
df['成交时间'] = pd.to_datetime(df['成交时间'])
df['日期'] = df['成交时间'].dt.date
df['小时'] = df['成交时间'].dt.hour
df['星期'] = df['成交时间'].dt.dayofweek
df['月份'] = df['成交时间'].dt.month
df['季度'] = df['成交时间'].dt.quarter

print('数据维度:', df.shape)
print('门店列表:', df['门店编号'].unique().tolist())
print('时间范围:', df['成交时间'].min(), '~', df['成交时间'].max())
print('订单数:', df['订单ID'].nunique())
df.head()


In [ ]:
quality = pd.DataFrame({'缺失值': df.isnull().sum(), '唯一值数': df.nunique()})
print('=== 数据质量 ===')
display(quality)

print('\n=== 各门店交易概况 ===')
store_summary = df.groupby('门店编号').agg(
    交易笔数=('销量', 'count'), 总销量=('销量', 'sum'), 平均单价=('单价', 'mean'),
    订单数=('订单ID', 'nunique'), 负销量笔数=('销量', lambda x: (x < 0).sum())
).round(3)
display(store_summary)

df['是否退货'] = df['销量'] < 0
print(f"退货/冲正记录: {df['是否退货'].sum()} 笔")


---## 二、节假日与季度对销量的影响分析

In [ ]:
CN_HOLIDAYS_2017 = {
    '2017-01-01': '元旦', '2017-01-02': '元旦假期',
    '2017-01-27': '春节', '2017-01-28': '春节', '2017-01-29': '春节',
    '2017-01-30': '春节', '2017-01-31': '春节', '2017-02-01': '春节', '2017-02-02': '春节',
    '2017-04-02': '清明节', '2017-04-03': '清明节', '2017-04-04': '清明节',
    '2017-05-01': '劳动节', '2017-05-28': '端午节', '2017-05-29': '端午节', '2017-05-30': '端午节',
    '2017-10-01': '国庆节', '2017-10-02': '国庆节', '2017-10-03': '国庆节',
    '2017-10-04': '国庆节', '2017-10-05': '国庆节', '2017-10-06': '国庆节', '2017-10-07': '国庆节',
}

df['日期_str'] = df['成交时间'].dt.strftime('%Y-%m-%d')
df['是否节假日'] = df['日期_str'].isin(CN_HOLIDAYS_2017.keys())
df['节假日名称'] = df['日期_str'].map(CN_HOLIDAYS_2017).fillna('非节假日')
df['是否节后工作日'] = (df['日期_str'] == '2017-01-03')
df['季度标签'] = df['季度'].map({1: 'Q1春季', 2: 'Q2夏季', 3: 'Q3秋季', 4: 'Q4冬季'})

weekday_names = ['一','二','三','四','五','六','日']
print('当前数据日期属性:')
print(f"  日期: {df['日期_str'].iloc[0]}")
print(f"  星期: 周{weekday_names[df['星期'].iloc[0]]}")
print(f"  季度: {df['季度标签'].iloc[0]}")
print(f"  节假日: {df['节假日名称'].iloc[0]}")
print(f"  节后首个工作日: {df['是否节后工作日'].iloc[0]}")


In [ ]:
hourly = df.groupby(['门店编号', '小时']).agg(
    销量=('销量', 'sum'), 均价=('单价', 'mean'),
    交易笔数=('销量', 'count'), 订单数=('订单ID', 'nunique')
).reset_index()
hourly['季度'] = 1
hourly['季度标签'] = 'Q1春季'
hourly['是否节后工作日'] = True

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for store in hourly['门店编号'].unique():
    sub = hourly[hourly['门店编号'] == store]
    axes[0].plot(sub['小时'], sub['销量'], marker='o', label=store, linewidth=2)
axes[0].set_title('各门店小时销量分布（节后工作日 2017-01-03）')
axes[0].set_xlabel('小时'); axes[0].set_ylabel('销量'); axes[0].legend()
axes[0].axvspan(7, 9, alpha=0.15, color='orange')
axes[0].axvspan(17, 20, alpha=0.15, color='green')

q1_hourly = hourly.groupby('小时')['销量'].sum().reset_index()
axes[1].bar(q1_hourly['小时'], q1_hourly['销量'], color='steelblue', alpha=0.8)
axes[1].set_title('Q1 季度 — 全门店合计小时销量')
axes[1].set_xlabel('小时'); axes[1].set_ylabel('总销量')
plt.tight_layout(); plt.show()

peak_hours = [8, 9, 10, 17, 18, 19]
hourly['时段类型'] = hourly['小时'].apply(lambda h: '高峰时段' if h in peak_hours else '非高峰时段')
period_effect = hourly.groupby(['门店编号', '时段类型'])['销量'].mean().unstack()
print('\n=== 高峰 vs 非高峰 平均小时销量 ===')
display(period_effect.round(2))
ratio = period_effect['高峰时段'].mean() / period_effect['非高峰时段'].mean()
print(f'\n高峰时段销量约为非高峰的 {ratio:.1f} 倍')


In [ ]:
daily_store = df.groupby(['门店编号', '日期']).agg(
    日销量=('销量', 'sum'), 日均价=('单价', 'mean'), 日订单数=('订单ID', 'nunique')
).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
stores = daily_store['门店编号'].unique()
colors = sns.color_palette('Set2', len(stores))
totals = daily_store.groupby('门店编号')['日销量'].sum()
bars = ax.bar(stores, totals, color=colors)
ax.set_title('各门店日总销量对比（Q1 · 节后工作日）')
ax.set_ylabel('日销量')
for bar, val in zip(bars, totals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, f'{val:.0f}', ha='center', fontsize=11)
plt.show()

print('\n【节假日与季度影响分析小结】')
print('1. 数据覆盖 2017-01-03（元旦后首个工作日，Q1 春季）')
print('2. 节后复工日呈现明显双峰结构：早高峰(8-10时) + 晚高峰(17-19时)')
print('3. CDXL 日销量最高，CDNL 最低，门店区位/规模差异显著')
print('4. Q1 年初消费偏刚需，单价中等偏低商品走量更快')


---## 三、销量与价格异常检测

In [ ]:
def detect_outliers_iqr(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    return (series < lower) | (series > upper), lower, upper

def detect_outliers_zscore(series, threshold=3):
    z = (series - series.mean()) / series.std()
    return np.abs(z) > threshold

for col_prefix, col in [('销量', '销量'), ('单价', '单价')]:
    for method in ['IQR', 'Z']:
        df[f'{col_prefix}_{method}异常'] = False

for store in df['门店编号'].unique():
    mask = df['门店编号'] == store
    for col in ['销量', '单价']:
        iqr_flag, _, _ = detect_outliers_iqr(df.loc[mask, col])
        z_flag = detect_outliers_zscore(df.loc[mask, col])
        df.loc[mask, f'{col}_IQR异常'] = iqr_flag.values
        df.loc[mask, f'{col}_Z异常'] = z_flag.values

df['销量_综合异常'] = df['销量_IQR异常'] | df['销量_Z异常']
df['单价_综合异常'] = df['单价_IQR异常'] | df['单价_Z异常']
df['综合异常'] = df['销量_综合异常'] | df['单价_综合异常']

anomaly_summary = pd.DataFrame({
    '指标': ['销量异常', '单价异常', '综合异常', '退货记录'],
    '笔数': [df['销量_综合异常'].sum(), df['单价_综合异常'].sum(), df['综合异常'].sum(), df['是否退货'].sum()],
    '占比(%)': [df['销量_综合异常'].mean()*100, df['单价_综合异常'].mean()*100, df['综合异常'].mean()*100, df['是否退货'].mean()*100]
}).round(2)
print('=== 交易级异常检测汇总 ===')
display(anomaly_summary)
display(df.groupby('门店编号').agg(销量异常=('销量_综合异常','sum'), 单价异常=('单价_综合异常','sum'), 综合异常=('综合异常','sum'), 总笔数=('销量','count')))


In [ ]:
features_if = ['单价', '销量', '小时']
X_scaled = StandardScaler().fit_transform(df[features_if])
iso = IsolationForest(contamination=0.05, random_state=42, n_estimators=100)
df['IF_异常'] = iso.fit_predict(X_scaled) == -1
print(f"Isolation Forest 检出异常: {df['IF_异常'].sum()} 笔 ({df['IF_异常'].mean()*100:.1f}%)")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for store in df['门店编号'].unique():
    sub = df[df['门店编号'] == store]
    axes[0,0].scatter(sub.loc[~sub['综合异常'],'单价'], sub.loc[~sub['综合异常'],'销量'], alpha=0.3, s=15, label=store)
    axes[0,0].scatter(sub.loc[sub['综合异常'],'单价'], sub.loc[sub['综合异常'],'销量'], marker='x', s=50, c='red')
axes[0,0].set_xlabel('单价'); axes[0,0].set_ylabel('销量'); axes[0,0].set_title('销量-单价散点图（红x=异常）'); axes[0,0].legend()
sns.boxplot(data=df, x='门店编号', y='单价', ax=axes[0,1]); axes[0,1].set_title('各门店单价分布')
sns.boxplot(data=df, x='门店编号', y='销量', ax=axes[1,0]); axes[1,0].set_title('各门店销量分布')
for store in hourly['门店编号'].unique():
    sub = hourly[hourly['门店编号'] == store]
    flag, _, _ = detect_outliers_iqr(sub['销量'])
    axes[1,1].plot(sub['小时'], sub['销量'], 'o-', label=store)
    axes[1,1].scatter(sub.loc[flag,'小时'], sub.loc[flag,'销量'], marker='*', s=150, zorder=5)
axes[1,1].set_title('小时销量异常点（★）'); axes[1,1].set_xlabel('小时'); axes[1,1].legend()
plt.tight_layout(); plt.show()

print('\n=== 典型异常交易 TOP10 ===')
display(df[df['综合异常'] | df['IF_异常']].nlargest(10, '销量')[['门店编号','单价','销量','成交时间','销量_综合异常','单价_综合异常','IF_异常','是否退货']])


---## 四、销量预测模型构建与门店对比

In [ ]:
def build_features(hourly_df):
    feat = hourly_df.copy().sort_values(['门店编号', '小时'])
    feat['小时_sin'] = np.sin(2 * np.pi * feat['小时'] / 24)
    feat['小时_cos'] = np.cos(2 * np.pi * feat['小时'] / 24)
    feat['是否高峰'] = feat['小时'].isin([8, 9, 10, 17, 18, 19]).astype(int)
    feat['是否节后工作日'] = 1
    feat['季度'] = 1
    feat['销量_lag1'] = feat.groupby('门店编号')['销量'].shift(1)
    feat['销量_lag1'] = feat['销量_lag1'].fillna(feat['销量'].mean())
    return feat

feat_df = build_features(hourly)
FEATURE_COLS = ['小时', '小时_sin', '小时_cos', '是否高峰', '是否节后工作日', '季度', '均价', '销量_lag1']
TARGET = '销量'
display(feat_df[['门店编号'] + FEATURE_COLS + [TARGET]].head(10))


In [ ]:
MODELS = {
    '线性回归': LinearRegression(),
    'Ridge回归': Ridge(alpha=1.0),
    '随机森林': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
    '梯度提升': GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
}

def evaluate_model(y_true, y_pred):
    return {'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2': r2_score(y_true, y_pred)}

results, predictions = [], {}
loo = LeaveOneOut()
for store in feat_df['门店编号'].unique():
    store_data = feat_df[feat_df['门店编号'] == store].reset_index(drop=True)
    X, y = store_data[FEATURE_COLS].values, store_data[TARGET].values
    for model_name, model in MODELS.items():
        y_pred_loo = np.zeros(len(y))
        for train_idx, test_idx in loo.split(X):
            m = model.__class__(**model.get_params())
            m.fit(X[train_idx], y[train_idx])
            y_pred_loo[test_idx] = m.predict(X[test_idx])
        metrics = evaluate_model(y, y_pred_loo)
        metrics.update({'门店': store, '模型': model_name})
        results.append(metrics)
        predictions[(store, model_name)] = (store_data['小时'].values, y, y_pred_loo)

results_df = pd.DataFrame(results)
pivot_mae = results_df.pivot(index='门店', columns='模型', values='MAE').round(2)
pivot_r2 = results_df.pivot(index='门店', columns='模型', values='R2').round(3)
print('=== 留一法交叉验证 MAE ==='); display(pivot_mae)
print('R2 得分:'); display(pivot_r2)
best_per_store = results_df.loc[results_df.groupby('门店')['MAE'].idxmin()]
print('各门店最优模型:'); display(best_per_store[['门店','模型','MAE','RMSE','R2']])


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, store in zip(axes, feat_df['门店编号'].unique()):
    best_model = best_per_store[best_per_store['门店']==store]['模型'].values[0]
    hours, y_true, y_pred = predictions[(store, best_model)]
    ax.plot(hours, y_true, 'o-', label='实际销量', linewidth=2, markersize=8)
    ax.plot(hours, y_pred, 's--', label=f'预测({best_model})', linewidth=2, markersize=6)
    ax.fill_between(hours, y_true, y_pred, alpha=0.2)
    ax.set_title(f'{store} — 最优: {best_model}'); ax.set_xlabel('小时'); ax.set_ylabel('销量'); ax.legend(fontsize=9)
plt.suptitle('各门店小时销量预测对比', fontsize=14, y=1.02); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot_mae, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax)
ax.set_title('各门店 x 各模型 MAE 热力图（越低越好）'); plt.tight_layout(); plt.show()


In [ ]:
print('=== 时段分割预测（训练 6-14时 -> 预测 15-21时）===\n')
split_hour = 14
holdout_results = []
for store in feat_df['门店编号'].unique():
    store_data = feat_df[feat_df['门店编号'] == store]
    train, test = store_data[store_data['小时'] <= split_hour], store_data[store_data['小时'] > split_hour]
    if len(test) == 0: continue
    for model_name, model in MODELS.items():
        model.fit(train[FEATURE_COLS], train[TARGET])
        y_pred = model.predict(test[FEATURE_COLS])
        m = evaluate_model(test[TARGET], y_pred)
        m.update({'门店': store, '模型': model_name})
        holdout_results.append(m)
display(pd.DataFrame(holdout_results).pivot(index='门店', columns='模型', values='MAE').round(2))
store_totals = feat_df.groupby('门店编号')['销量'].sum().sort_values(ascending=False)
print('\n门店日销量排名:')
for rank, (store, vol) in enumerate(store_totals.items(), 1):
    print(f'  {rank}. {store}: {vol:.1f}')


---## 五、综合结项报告

In [ ]:
total_sales = df['销量'].sum()
total_orders = df['订单ID'].nunique()
anomaly_rate = df['综合异常'].mean() * 100
best_models = best_per_store.set_index('门店')['模型'].to_dict()
top_store = store_totals.index[0]
ratio = period_effect['高峰时段'].mean() / period_effect['非高峰时段'].mean()

report = f"""
================================================================
           门店销量预测与异常监控报告 — 结项报告
           报告日期: {REPORT_DATE}
================================================================

一、项目概述
  基于 data3.csv（{df['成交时间'].min().date()}，{df['门店编号'].nunique()} 家门店、
  {len(df)} 笔交易、{total_orders} 个订单），完成影响因素分析、异常检测与预测建模。

二、数据概况
  门店: {', '.join(df['门店编号'].unique())}
  总销量: {total_sales:.1f}  |  订单数: {total_orders}
  时间: 单日（元旦后首个工作日，Q1春季）

三、节假日与季度影响
  1. 节后工作日呈早高峰(8-10时)+晚高峰(17-19时)双峰，高峰约为非高峰 {ratio:.1f} 倍
  2. Q1 刚需消费为主，CDXL({store_totals.get('CDXL',0):.0f}) > CDLG({store_totals.get('CDLG',0):.0f}) > CDNL({store_totals.get('CDNL',0):.0f})

四、异常检测
  销量异常: {df['销量_综合异常'].sum()} 笔 | 单价异常: {df['单价_综合异常'].sum()} 笔
  综合异常率: {anomaly_rate:.1f}% | 退货: {df['是否退货'].sum()} 笔

五、销量预测（留一法交叉验证）
"""
for store, model in best_models.items():
    row = best_per_store[best_per_store['门店']==store].iloc[0]
    report += f"  {store}: {model} (MAE={row['MAE']:.1f}, R2={row['R2']:.3f})\n"

report += f"""
  销量冠军: {top_store}（{store_totals[top_store]:.0f}）

六、业务建议
  1. 按双峰规律在 8-10 时、17-19 时增派人手
  2. 对高价/负销量交易实时告警
  3. CDNL 销量偏低，建议分析品类与促销
  4. 部署小时级销量预测支撑备货调度
  5. 建议积累 >=30 天数据验证节假日/季度效应

================================================================
"""
print(report)
with open('门店销量预测与异常监控报告.txt', 'w', encoding='utf-8') as f:
    f.write(report)
print('\n报告已保存: 门店销量预测与异常监控报告.txt')


---## 附录：关键结论速览| 维度 | 关键发现 ||------|----------|| **节假日** | 元旦后首个工作日，早/晚双峰明显 || **季度** | Q1 刚需消费，CDXL 领先 || **异常** | 约 5% 交易存在价格或销量异常 || **预测** | 树模型在小时级预测表现较优 || **建议** | 建立实时异常监控 + 小时级销量预测系统 |